In [7]:
from google.colab import files
files.upload()

Saving dataset.jsonl to dataset (1).jsonl


{'dataset (1).jsonl': b'{"instruction": "What is RAG?", "output": "RAG (Retrieval-Augmented Generation) combines document retrieval with LLM generation to produce more accurate and grounded answers."}\r\n{"instruction": "What is fine-tuning?", "output": "Fine-tuning is the process of training a pre-trained model on a specific dataset to adapt it to a task or domain."}\r\n{"instruction": "What is LoRA?", "output": "LoRA is a parameter-efficient fine-tuning method that trains small adapter layers instead of updating the full model."}\r\n{"instruction": "What is an LLM?", "output": "An LLM is a large language model trained on massive text data to generate human-like responses."}\r\n{"instruction": "What is an embedding?", "output": "An embedding is a numerical vector representation of text used to measure semantic similarity."}\r\n\r\n{"instruction": "What is a vector database?", "output": "A vector database stores embeddings and enables fast similarity search for RAG systems."}\r\n{"inst

In [8]:
from datasets import load_dataset

dataset_path = "/content/dataset.jsonl"

dataset = load_dataset(
    "json",
    data_files=dataset_path,
    split="train"
)

print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

{'instruction': 'What is RAG?', 'output': 'RAG (Retrieval-Augmented Generation) combines document retrieval with LLM generation to produce more accurate and grounded answers.'}


In [9]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-3-mini-4k-instruct")
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    text = f"Instruction: {example['instruction']}\nAnswer: {example['output']}"

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

print(tokenized_dataset[0])

NameError: name 'AutoTokenizer' is not defined

In [11]:
from transformers import AutoTokenizer

In [12]:
model_name = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Tokenizer loaded


In [13]:
print(tokenizer)

TokenizersBackend(name_or_path='microsoft/phi-3-mini-4k-instruct', vocab_size=32000, model_max_length=4096, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=False),
	32000: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<|assistant|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<|placeholder1|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<|placeholder2|>", rstrip=

In [14]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="dataset.jsonl",
    split="train"
)

print(dataset[0])
print("Rows:", len(dataset))

{'instruction': 'What is RAG?', 'output': 'RAG (Retrieval-Augmented Generation) combines document retrieval with LLM generation to produce more accurate and grounded answers.'}
Rows: 45


In [15]:
def tokenize(example):
    text = f"Instruction: {example['instruction']}\nAnswer: {example['output']}"

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

tokenized_dataset = tokenized_dataset.remove_columns(
    ["instruction", "output"]
)

print(tokenized_dataset[0].keys())

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [16]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [17]:
import torch
from transformers import AutoModelForCausalLM

model_name = "microsoft/phi-3-mini-4k-instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded on device")
print("Device:", model.hf_device_map)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded on device


AttributeError: 'Phi3ForCausalLM' object has no attribute 'hf_device_map'

In [18]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["o_proj"]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [19]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [1]:
!pip install -U transformers datasets peft accelerate torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.9.0
    Uninstalling transformers-5.9.0:
      Successfully uninstalled transformers-5.9.0


In [1]:
!pip install -U torchao

In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

In [2]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [3]:
dataset = load_dataset(
    "json",
    data_files="dataset.jsonl",
    split="train"
)

print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

{'instruction': 'What is RAG?', 'output': 'RAG (Retrieval-Augmented Generation) combines document retrieval with LLM generation to produce more accurate and grounded answers.'}


In [4]:
model_name = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
def tokenize(example):
    text = f"Instruction: {example['instruction']}\nAnswer: {example['output']}"

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


tokenized_dataset = dataset.map(tokenize)
tokenized_dataset = tokenized_dataset.remove_columns(["instruction", "output"])

print(tokenized_dataset[0])

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

{'input_ids': [32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 2799, 4080, 29901, 1724, 338, 390, 10051, 29973, 13, 22550, 29901, 390, 10051, 313, 8015, 2546, 791, 29899, 29909, 688, 358, 287, 28203, 29897, 4145, 1475, 1842, 5663, 16837, 411, 365, 26369, 12623, 304, 7738, 901, 16232, 322, 5962, 287, 6089, 29889], 'attention_mask': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [6]:
import torch
from transformers import AutoModelForCausalLM

model_name = "microsoft/phi-3-mini-4k-instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully")
print("Device map:", model.hf_device_map if hasattr(model, "hf_device_map") else "no device_map attribute")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Model loaded successfully
Device map: no device_map attribute


In [7]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["o_proj"]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 3,822,652,416 || trainable%: 0.0411


In [11]:
trainer.train()

NameError: name 'trainer' is not defined

In [12]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

In [13]:
model_name = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [14]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="dataset.jsonl",
    split="train"
)

In [16]:
def tokenize(example):
    text = f"Instruction: {example['instruction']}\nAnswer: {example['output']}"

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)
tokenized_dataset = tokenized_dataset.remove_columns(["instruction", "output"])

In [17]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [18]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 3,822,652,416 || trainable%: 0.0411


In [22]:
training_args = TrainingArguments(
    output_dir="./lora-output",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=5,
    save_steps=50,
    report_to="none",
    fp16=True
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer ready")

[transformers] The model is already on multiple devices. Skipping the move to device specified in `args`.


Trainer ready


In [23]:
trainer.train()

RuntimeError: Function MmBackward0 returned an invalid gradient at index 1 - expected device meta but got cuda:0

In [24]:
!pip install -q transformers datasets accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00


In [26]:
import torch
from transformers import AutoModelForCausalLM

model_name = "microsoft/phi-3-mini-4k-instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [27]:
model = model.to("cuda")

print("Model moved to:", next(model.parameters()).device)

OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 17.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.34 GiB is allocated by PyTorch, and 70.75 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [28]:
!pip install bitsandbytes accelerate

In [29]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "microsoft/phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [32]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["o_proj"]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 3,822,652,416 || trainable%: 0.0411


In [33]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./lora-output",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=5,
    save_steps=50,
    report_to="none",
    fp16=True
)

In [34]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [35]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer ready")

Trainer ready


In [36]:
trainer.train()

Step,Training Loss
5,3.089549
10,3.252489
15,2.947694
20,2.590363


TrainOutput(global_step=23, training_loss=2.937289942865786, metrics={'train_runtime': 15.5612, 'train_samples_per_second': 2.892, 'train_steps_per_second': 1.478, 'total_flos': 128706686484480.0, 'train_loss': 2.937289942865786, 'epoch': 1.0})